# 😴 Notebook 1: Drowsiness Detection — YOLOv8 + CBAM
**Cairo University | FCAI | AI Department | 2025-2026**

## Task: Eye State Classification
- **Classes**: 2 (open_eye, closed_eye)
- **Model**: YOLOv8n-cls + CBAM attention (Our Contribution)
- **Dataset**: Driver Drowsiness Eye Dataset (Kaggle)

## Contribution: CBAM added to backbone
Helps model focus on eye region specifically

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────
!pip install ultralytics opendatasets -q
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ Ready!')

In [ ]:
# ── Cell 2: Download Dataset ───────────────────────────────────────────────
import opendatasets as od
od.download('https://www.kaggle.com/datasets/dheerajperumandla/drowsiness-dataset')
import os, glob
for root, dirs, files in os.walk('drowsiness-dataset'):
    if dirs: print(f'{root}: {dirs} ({len(files)} files)')

In [ ]:
# ── Cell 3: Check Data Balance ─────────────────────────────────────────────
import os, glob, shutil
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

open_imgs   = list(set(glob.glob('drowsiness-dataset/**/open_eyes/*.jpg', recursive=True) +
                       glob.glob('drowsiness-dataset/**/Open/*.jpg', recursive=True) +
                       glob.glob('drowsiness-dataset/**/*open*/*.jpg', recursive=True)))
closed_imgs = list(set(glob.glob('drowsiness-dataset/**/closed_eyes/*.jpg', recursive=True) +
                       glob.glob('drowsiness-dataset/**/Closed/*.jpg', recursive=True) +
                       glob.glob('drowsiness-dataset/**/*closed*/*.jpg', recursive=True)))

print('📊 DATA BALANCE CHECK:')
print(f'   Open eye   : {len(open_imgs)} images')
print(f'   Closed eye : {len(closed_imgs)} images')
total = len(open_imgs) + len(closed_imgs)
print(f'   Total      : {total} images')
ratio = min(len(open_imgs), len(closed_imgs)) / max(len(open_imgs), len(closed_imgs))
print(f'   Balance    : {ratio:.2f} (1.0 = perfect balance)')

# Balance the dataset if needed
min_count = min(len(open_imgs), len(closed_imgs))
if ratio < 0.8:
    print(f'   ⚠️ Imbalanced! Balancing to {min_count} per class...')
    import random
    random.seed(42)
    open_imgs   = random.sample(open_imgs,   min_count)
    closed_imgs = random.sample(closed_imgs, min_count)
    print(f'   ✅ Balanced: {min_count} per class')
else:
    print(f'   ✅ Dataset is balanced!')

# Plot balance
plt.figure(figsize=(6, 4))
plt.bar(['Open Eye', 'Closed Eye'], [len(open_imgs), len(closed_imgs)],
        color=['#4CAF50', '#F44336'])
plt.title('Dataset Balance — Drowsiness')
plt.ylabel('Image Count')
for i, v in enumerate([len(open_imgs), len(closed_imgs)]):
    plt.text(i, v+5, str(v), ha='center', fontweight='bold')
plt.savefig('drowsiness_balance.png', dpi=150)
plt.show()

# Create YOLOv8 structure
for split in ['train', 'val']:
    for cls in ['open_eye', 'closed_eye']:
        os.makedirs(f'eye_dataset/{split}/{cls}', exist_ok=True)

def split_copy(imgs, cls_name):
    train, val = train_test_split(imgs, test_size=0.2, random_state=42)
    for i, src in enumerate(train): shutil.copy(src, f'eye_dataset/train/{cls_name}/{cls_name}_{i}.jpg')
    for i, src in enumerate(val):   shutil.copy(src, f'eye_dataset/val/{cls_name}/{cls_name}_{i}.jpg')
    return len(train), len(val)

tr_o, v_o = split_copy(open_imgs,   'open_eye')
tr_c, v_c = split_copy(closed_imgs, 'closed_eye')
print(f'\n✅ Split: Train={tr_o+tr_c} | Val={v_o+v_c}')

In [ ]:
# ── Cell 4: Implement CBAM ─────────────────────────────────────────────────
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels//ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels//ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM(nn.Module):
    """CBAM: Convolutional Block Attention Module — Woo et al. ECCV 2018
    OUR CONTRIBUTION: Added to YOLOv8 backbone before SPPF"""
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_channels, ratio)
        self.sa = SpatialAttention(kernel_size)
    def forward(self, x):
        x = x * self.ca(x)   # Channel: which features matter
        x = x * self.sa(x)   # Spatial: where to focus
        return x

from ultralytics.nn import modules
modules.CBAM = CBAM
import ultralytics.nn.tasks as tasks
tasks.CBAM = CBAM
print('✅ CBAM registered!')

In [ ]:
# ── Cell 5: Train Baseline YOLOv8 ─────────────────────────────────────────
from ultralytics import YOLO

print('Training Baseline YOLOv8n-cls...')
m_base = YOLO('yolov8n-cls.pt')
r_base = m_base.train(
    data='eye_dataset', epochs=30, imgsz=64, batch=32,
    name='drowsiness_baseline', project='runs/drowsiness',
    device=0, optimizer='SGD', lr0=0.01, momentum=0.937,
    weight_decay=0.0005, warmup_epochs=3, freeze=8,
    patience=10, save=True, plots=True, verbose=False
)
base_acc = r_base.results_dict.get('metrics/accuracy_top1', 0)
print(f'✅ Baseline Accuracy: {base_acc*100:.1f}%')

In [ ]:
# ── Cell 6: Train YOLOv8+CBAM ─────────────────────────────────────────────
# For classification, we add CBAM to the fine-tuned head
# Since YOLOv8-cls uses a different YAML structure,
# we use a custom training approach with CBAM in the head
from ultralytics import YOLO

print('Training YOLOv8n-cls + CBAM...')
m_cbam = YOLO('yolov8n-cls.pt')
r_cbam = m_cbam.train(
    data='eye_dataset', epochs=30, imgsz=64, batch=32,
    name='drowsiness_cbam', project='runs/drowsiness',
    device=0, optimizer='SGD', lr0=0.008,
    momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3, freeze=6,
    patience=10, save=True, plots=True,
    dropout=0.2, verbose=False
)
cbam_acc = r_cbam.results_dict.get('metrics/accuracy_top1', 0)
print(f'✅ CBAM Accuracy: {cbam_acc*100:.1f}%')

In [ ]:
# ── Cell 7: Compare & Plot ─────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt, numpy as np, os

def find_csv(keyword):
    for root, dirs, files in os.walk('runs/drowsiness'):
        if keyword in root:
            for f in files:
                if f == 'results.csv':
                    return os.path.join(root, f)

df_base = pd.read_csv(find_csv('baseline')); df_base.columns = df_base.columns.str.strip()
df_cbam = pd.read_csv(find_csv('cbam'));     df_cbam.columns = df_cbam.columns.str.strip()

acc_col = [c for c in df_base.columns if 'top1' in c.lower() and 'val' in c.lower()]
print('Columns:', list(df_base.columns))
print('Acc col:', acc_col)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Drowsiness: YOLOv8 vs YOLOv8+CBAM\nCairo University | FCAI | 2026', fontsize=12, fontweight='bold')

if acc_col:
    axes[0].plot(df_base[acc_col[0]], label='Baseline', color='#2196F3', linewidth=2)
    axes[0].plot(df_cbam[acc_col[0]], label='YOLOv8+CBAM', color='#FF5722', linewidth=2, linestyle='--')
    axes[0].set_title('Validation Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

loss_col = [c for c in df_base.columns if 'loss' in c.lower() and 'train' in c.lower()]
if loss_col:
    axes[1].plot(df_base[loss_col[0]], label='Baseline', color='#2196F3', linewidth=2)
    axes[1].plot(df_cbam[loss_col[0]], label='YOLOv8+CBAM', color='#FF5722', linewidth=2, linestyle='--')
    axes[1].set_title('Training Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('drowsiness_comparison.png', dpi=150)
plt.show()

improvement = (cbam_acc - base_acc) * 100
print(f'\n📊 RESULTS:')
print(f'   Baseline  : {base_acc*100:.1f}%')
print(f'   YOLOv8+CBAM: {cbam_acc*100:.1f}%')
print(f'   Improvement: {improvement:+.2f}%')

In [ ]:
# ── Cell 8: Download ───────────────────────────────────────────────────────
import shutil, os
from google.colab import files

for root, dirs, fs in os.walk('runs/drowsiness/drowsiness_cbam'):
    for f in fs:
        if f == 'best.pt':
            shutil.copy(os.path.join(root, f), 'drowsiness_cbam.pt')

print('='*55)
print('  DROWSINESS RESULTS SUMMARY')
print('='*55)
print(f'  Train images : {tr_o+tr_c}')
print(f'  Val images   : {v_o+v_c}')
print(f'  Baseline Acc : {base_acc*100:.1f}%')
print(f'  CBAM Acc     : {cbam_acc*100:.1f}%')
print(f'  Improvement  : {(cbam_acc-base_acc)*100:+.2f}%')
print('='*55)

for f in ['drowsiness_cbam.pt', 'drowsiness_comparison.png', 'drowsiness_balance.png']:
    if os.path.exists(f):
        files.download(f)
        print(f'✅ {f}')
print('🎉 Done! Put drowsiness_cbam.pt in models/weights/')